In [ ]:
import sys
sys.path.append("../")

import numpy as np

# Generalized Second-Price (GSP) Auction

This notebook illustrates how to use SODA to compute a Bayes-Nash equilibrium
for a **Generalized Second-Price (GSP)** keyword auction.

**Setting**
- 5 symmetric bidders, values drawn i.i.d. from Uniform[0, 1]
- 3 ad slots with click-through rates α = (1.0, 0.8, 0.6)
- A zero-CTR position pads the implementation's required 4-position vector
- No reserve price
- Bidder ranked *k* (0-indexed) wins slot *k*, receives CTR α_k, and pays the
  bid of the next-ranked bidder

Unlike a second-price (Vickrey) auction, GSP is **not** incentive-compatible:
truthful bidding is generally not an equilibrium.  SODA learns the symmetric
BNE numerically.

## Mechanism, Game, and Learner

In [ ]:
from soda.util.config import Config

config_game    = "../configs/game/gsp/gsp.yaml"
config_learner = "../configs/learner/sofw.yaml"

config = Config(config_game, config_learner)
game, learner = config.create_setting()
game.get_utility()

In [ ]:
print(game.mechanism)

In [ ]:
print(learner)

### Quick utility check

With 5 bidders submitting bids (0.9, 0.7, 0.5, 0.3, 0.1) and bidder 0 having value 1.0:
- Bidder 0 wins slot 0 (CTR = 1.0) and pays max(reserve, 0.7) = 0.7
- Utility = 1.0 × (1.0 − 0.7) = 0.3

In [ ]:
obs_profile = np.array([[1.0], [0.8], [0.6], [0.4], [0.2]])
bids_profile = np.array([[0.9], [0.7], [0.5], [0.3], [0.1]])

u = game.mechanism.utility(obs_profile, bids_profile, index_bidder=0)
print(f"Utility of top bidder (value=1.0, bid=0.9, next bid=0.7): {u[0]:.4f}")
assert np.isclose(u[0], 1.0 * (1.0 - 0.7)), "utility check failed"

## Learning the BNE

In [ ]:
strategies = config.create_strategies(game, init_method="random")
learner.run(game, strategies, print_result=True, save_history=True)

## Results

The strategy plot shows the learned bidding distribution: each row is a value
level, each column is a bid level, and darkness indicates probability mass.
At BNE the distribution concentrates along a monotone bid function — bidders
with higher values bid more, but shade below their true value.

In [ ]:
strategies['1'].plot(grad=True, metrics=True)

### Equilibrium bid function

Extract the mode bid for each discrete value level to recover the approximate
equilibrium bid function β(v), and compare it to truthful bidding β(v) = v.

In [ ]:
import matplotlib.pyplot as plt

strat = strategies['1']
# modal bid at each value level
bid_modes = strat.a_discr[strat.x.argmax(axis=1)]

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(strat.o_discr, bid_modes, label="SODA BNE", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Truthful (β=v)")
ax.set_xlabel("Value v")
ax.set_ylabel("Bid β(v)")
ax.set_title("GSP equilibrium bid function (5 bidders, 3 slots)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Convergence over iterations

In [ ]:
iter_snapshot = 200
strategies['1'].plot(grad=True, metrics=True, iter=iter_snapshot)

In [ ]:
t = iter_snapshot
print(f"After {t} iterations:")
print(f"  Utility:      {strategies['1'].utility[t]:.5f}")
print(f"  Utility loss: {strategies['1'].utility_loss[t]:.5f}")